# Adidas LAM Chatbot Analytics

This notebook reproduces the core business-case analysis from the supplied Excel workbook. The workbook supports volume, channel, category, subcategory, classification-failure, and prioritization analysis. The containment, repeat-contact, and resolution KPIs are treated as business-case inputs because the workbook does not contain session IDs, customer IDs, timestamps, CSAT, or resolution outcome labels.

# Section 1 — Imports

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np



# Section 2 — Configuration

In [4]:
DATA_PATH = Path(r"C:\Users\restr\Desktop\adidas-chatbot-case\data\raw\Business_Case_Chatbot_data_Raw_Data.xlsx")

if DATA_PATH.exists():
    print(f" Success! File found at: {DATA_PATH}")
else:
    print(f" ERROR: File NOT found at: {DATA_PATH}")
    print("Please double-check the folder path or spelling.")

 Success! File found at: C:\Users\restr\Desktop\adidas-chatbot-case\data\raw\Business_Case_Chatbot_data_Raw_Data.xlsx


# Section 3 — Cargar Dataset

In [5]:
# Abrir el archivo de Excel para inspeccionar la metadata
xls = pd.ExcelFile(DATA_PATH)

# Mostrar los nombres de las hojas del archivo
print("   Hojas disponibles en el archivo de excel:")
for index, name in enumerate(xls.sheet_names, start=1):
    print(f"  {index}. {name}")

   Hojas disponibles en el archivo de excel:
  1. Agent handled only volume
  2. Hybrid Handled only volume
  3. Bot only volume


# Section 4 — Cargar cada hoja

1. Agent handled only volume --> df_agent
2. Hybrid Handled only volume --> df_hybrid
3. Bot only volume --> df_bot

In [6]:
# Seleccionamos la primera hoja dinámicamente de los metadatos.
df_agent = pd.read_excel(DATA_PATH, sheet_name="Agent handled only volume")
df_hybrid = pd.read_excel(DATA_PATH, sheet_name="Hybrid Handled only volume")
df_bot = pd.read_excel(DATA_PATH, sheet_name="Bot only volume")

Lo anterior lo segmentamos por:

- modelo de gestión
- ruta de escalamiento
- nivel de automatización

Esto significa que Adidas realiza un seguimiento operativo de las conversaciones según el canal de resolución de problemas.


Lo que probablemente representa cada hoja

| Hoja | Significado |
| :--- | :--- |
| Agent handled only volume | Interacciones solo con humanos |
| Hybrid Handled only volume  | Bot + escalamiento humano |
| Bot only volume | Interacciones totalmente automatizadas |

## Por qué esto es extremadamente importante

Esta estructura nos permite analizar:

### A. Eficacia de la automatización
Podemos comparar:
- Lo que el bot resuelve **por sí solo**.
- Lo que requiere **escalamiento**.
- Lo que **evita** la automatización por completo.

### B. Complejidad de la intención/petición
Algunas intenciones/peticiones son:
- Fáciles de automatizar.
- Parcialmente automatizables.
- Imposibles de automatizar de forma segura.

Esta segmentación ayuda a identificarlas.

### C. Fugas de escalamiento
Los flujos híbridos son especialmente importantes porque suelen indicar:
1. Que el bot gestionó parcialmente la solicitud,
2. pero no la resolvió por completo.

> **Nota:** Este es uno de los mayores costes operativos en los sistemas de IA conversacional.

## Perspectiva/Insight estratégica

En última instancia, todo se reduce a:

> **“¿Cómo reducimos las escaladas híbridas innecesarias?”**

Ese es probablemente el verdadero objetivo empresarial.

**Un Insight muy importante.**

# Section 5 — Inspect Shapes

## Agent handled only volume

In [7]:
df_agent.shape

(184, 8)

In [8]:
df_agent.head()

,Contact Reason,Conteo de Filas,Percentage,Unnamed: 3,Contact Reason.1,Sub Category,Conteo de Filas.1,Percentage.1
0,Returns & Refunds,116308.0,0.292082,NaN,Customer Feedback,Left Blank,70731,0.177625
1,Existing Order,106339.0,0.267047,NaN,Support on Ordering,Explain how to order,52148,0.130958
2,Customer Feedback,71907.0,0.180579,NaN,Returns & Refunds,Left Blank,36494,0.091647
3,Support on Ordering,52426.0,0.131656,NaN,Returns & Refunds,Return status,30817,0.077390
4,Spam/No Contact,16204.0,0.040693,NaN,Existing Order,Size,20702,0.051989


In [9]:
df_agent.info()

<class 'pandas.DataFrame'>
RangeIndex: 184 entries, 0 to 183
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Contact Reason     16 non-null     str    
 1   Conteo de Filas    18 non-null     float64
 2   Percentage         18 non-null     float64
 3   Unnamed: 3         0 non-null      float64
 4   Contact Reason.1   184 non-null    str    
 5   Sub Category       184 non-null    str    
 6   Conteo de Filas.1  184 non-null    int64  
 7   Percentage.1       184 non-null    float64
dtypes: float64(4), int64(1), str(3)
memory usage: 11.6 KB


### Interpretación de la Estructura de Datos

El archivo analizado presenta las siguientes dimensiones estructurales:
* **184** registros (filas).
* **8** atributos (columnas).

**Premisa analítica fundamental:**
El conjunto de datos **NO** contiene microdatos o registros a nivel de sesión conversacional (unstructured/raw conversation logs). En su lugar, estamos ante un **dataset agregado de performance operacional**

Si tuviéramos acceso a datos transaccionales crudos a nivel de conversación, la estructura esperada reflejaría el siguiente esquema conceptual:

| conversation_id | timestamp  | customer_id | message              | intent       | resolution |
| --------------- | ---------- | ----------- | -------------------- | ------------ | ---------- |
| 92813           | 2026-01-01 | C182        | “Where is my order?” | order_status | resolved   |

Es decir, una arquitectura de datos granular caracterizada por:
- Una fila equivalente a una interacción o mensaje único.
- Timestamps de alta resolución temporal.
- Identificadores únicos (IDs de usuario, sesión e interacción).
- Metadatos conversacionales avanzados y textos libres (raw text strings).

Por el contrario, el dataset actual expone estructuras consolidadas como la siguiente:

| Contact Reason    | Conteo de Filas |
| ----------------- | --------------: |
| Returns & Refunds |          116308 |

La interpretación estadística correcta de este registro es: **“Se registraron 116,308 interacciones indexadas bajo la taxonomía de Returns & Refunds”**. No estamos auditando el flujo conversacional en vivo, sino su volumetría histórica etiquetada.

La presencia explícita de la métrica/campo `"Conteo de Filas"` confirma que el origen es un cubo de datos o reporte pre-agregado. Como consecuencia directa:
* **Ausencia de granularidad:** No existen identificadores (IDs) ni marcas de tiempo (timestamps).
* **Ausencia de texto libre:** No se dispone del registro literalizado de los chats ni de metadatos transaccionales crudos.
* **Información resumida:** El dataset provee métricas descriptivas consolidadas (Conteos, Porcentajes) y taxonomías categóricas ya normalizadas.

>  **Impacto Metodológico:** Esta restricción en la granularidad de los datos redefine por completo nuestra estrategia analítica y el alcance del proyecto.

---

### Redefinición del Enfoque Técnico

Dadas las características de la data disponible, se delimita estrictamente el alcance de la solución:

#### Lo que NO es viable abordar: 
* Entrenamiento, fine-tuning o reentrenamiento de modelos NLP.
* Generación de embeddings semánticos o clustering de texto no estructurado.
* Análisis de sentimiento o minería de transcripciones.
* Optimización directa de modelos de lenguaje natural (LLM).

#### Lo que SÍ se abordará:
* **Analítica Operacional de Contact Centers:** Diagnóstico de fallas en el funnel de atención.
* **Auditoría de KPIs Críticos:** Evaluación causal de las brechas en Contención, Repetitividad y Resolución (Containment, Repeat, Resolution).
* **Modelado de Oportunidades:** Matriz de priorización para automatización basada en el volumen de fuga hacia canales humanos.

#### Anatomía del Funnel de Atención Conversacional

```
[ Entrada Total de Contactos ]
              │
              ▼
   ┌─────────────────────┐
   │ 1. Clasificación    │  (¿El bot entiende el intent/motivo?)
   └──────────┬──────────┘
              │
              ▼
   ┌─────────────────────┐
   │ 2. Contención       │  (¿El bot retiene el caso sin derivar?)
   └──────────┬──────────┘
              │
              ▼
   ┌─────────────────────┐
   │ 3. Resolución       │  (¿El problema realmente se solucionó?)
   └─────────────────────┘

```

1. Capa de Entrada y Clasificación (Input & Intent Capture)
   - **Qué es:** El punto de partida donde el usuario interactúa y expresa su motivo de contacto (ej. "¿Dónde está mi pedido?").

   - **Métrica Clave:** Tasa de Clasificación de Intents. Evalúa el porcentaje de conversaciones donde el sistema de procesamiento de lenguaje natural (NLP) logra encasillar la duda en una categoría específica.

   - **Punto de Falla Típico:** Fugas por etiquetas como "Left Blank" o "Not defined by Bot". Si esta capa falla, el usuario cae directamente a las etapas inferiores de manera desordenada.

2. Capa de Contención (Deflection / Automation Layer)
   - **Qué es:** El filtro donde el chatbot procesa la solicitud utilizando flujos automatizados (informacionales o transaccionales) para evitar que la interacción requiera un costo operativo humano.

   - **Métrica Clave:** Containment Rate (Tasa de Contención). El porcentaje de contactos que completan su ciclo dentro del bot sin ser transferidos a un agente/asesor humano.

   - **Dinámica Operacional:** Una alta contención reduce la presión sobre el contact center, pero no es sinónimo de éxito si se fuerza al cliente a salir del canal sin una respuesta real.

3. Capa de Resolución (Fulfillment & Outcome)
   - **Qué es:** El fondo del embudo. Es la confirmación de que la necesidad del cliente fue satisfecha de manera efectiva en su primer contacto.

   - **Métrica Clave:** Resolution Rate (Tasa de Resolución) / First Contact Resolution (FCR).

   - **El Enlace Causal:** Si la tasa de resolución es baja, se genera una ruptura en el funnel. Esto provoca un rebote que infla artificialmente la Tasa de Repetitividad (Repeat Rate), obligando al usuario a volver a entrar al embudo y destruyendo la eficiencia del ecosistema.


## Hybrid Handled only volume

In [10]:
df_hybrid.shape

(208, 8)

In [11]:
df_hybrid.head()

,Contact Reason,Conteo de Filas,Percentage,Unnamed: 3,Contact Reason.1,Sub Category,Conteo de Filas.1,Percentage.1
0,Existing Order,113898.0,0.411416,NaN,Spam/No Contact,Left Blank,26688,0.096401
1,Returns & Refunds,74494.0,0.269083,NaN,Existing Order,Size,20571,0.074305
2,Spam/No Contact,26788.0,0.096762,NaN,Returns & Refunds,Return status,18098,0.065373
3,Payment,13149.0,0.047496,NaN,Existing Order,In transit,14978,0.054103
4,Product Information,11036.0,0.039864,NaN,Returns & Refunds,label Request,14028,0.050671


In [12]:
df_hybrid.info()

<class 'pandas.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Contact Reason     19 non-null     str    
 1   Conteo de Filas    19 non-null     float64
 2   Percentage         19 non-null     float64
 3   Unnamed: 3         0 non-null      float64
 4   Contact Reason.1   208 non-null    str    
 5   Sub Category       208 non-null    str    
 6   Conteo de Filas.1  208 non-null    int64  
 7   Percentage.1       208 non-null    float64
dtypes: float64(4), int64(1), str(3)
memory usage: 13.1 KB


### Interpretación
Al analizar las matrices dimensionales de las hojas categóricas por canal, se observa una asimetría en el volumen de registros (*rows*):
* **Agent Only (Solo Humano):** Matrix de dimensión **(184, 8)** $\rightarrow$ 184 filas.
* **Hybrid (Bot + Humano):** Matrix de dimensión **(208, 8)** $\rightarrow$ 208 filas.

$$\text{Registros en Canal Híbrido (208)} > \text{Registros en Solo Agente (184)}$$

#### Formulación de Hipótesis Operacionales
Dado que este dataset se compone de datos agregados basados en taxonomías de contacto, una mayor cantidad de filas no implica mayor volumen de transacciones, sino una mayor cantidad de combinaciones de categorías exclusivas. A partir de esta asimetría estadística, se proponen dos hipótesis principales:

##### **Hipótesis 1:** Mayor Dispersión y Diversidad de la Intención (*Intent Diversity*)
Una tabla paramétrica con más registros indica una cobertura más amplia de escenarios de negocio. Esto sugiere que los usuarios que interactúan con el chatbot de forma inicial terminan cubriendo un espectro más diverso de problemáticas, subcategorías y combinaciones operativas que aquellos que se derivan o ingresan directamente con un agente humano.

##### **Hipótesis 2:** Alta Fragmentación Taxonómica en la Escalación
El canal híbrido muestra un fenómeno de **fragmentación taxonómica**. Mientras que el canal atendido puramente por agentes humanos opera bajo una clasificación más consolidada o simplificada, el flujo híbrido (donde el bot inicia y el humano finaliza) genera más subcategorías, *edge cases* o flujos de excepción. 

---

### Análisis de Complejidad: ¿Qué implica la "Fragmentación Taxonómica"?
La fragmentación taxonómica se traduce en una mayor granularidad y subdivisión del árbol de decisiones. 

* **Estructura Consolidada (Baja Complejidad):**
  `Returns` $\rightarrow$ `Orders` $\rightarrow$ `Payments`
* **Estructura Fragmentada / Canal Híbrido (Alta Complejidad):**
  - `Returns` $\rightarrow$ `Delayed Refund`
  - `Returns` $\rightarrow$ `Missing Label`
  - `Returns` $\rightarrow$ `Damaged Product`
  - `Returns` $\rightarrow$ `Refund Pending`

* Más granularidad.
* Más subdivisiones.
* Más complejidad operacional.



## Bot only volume

In [13]:
df_bot.shape

(77, 8)

In [14]:
df_bot.head()

,Contact Reason,Conteo de Filas,Percentage,Unnamed: 3,Contact Reason.1,Sub Category,Conteo de Filas.1,Percentage.1
0,Returns & Refunds,54282.0,0.439281,NaN,Returns & Refunds,How to return,19915,0.161164
1,Existing Order,50294.0,0.407008,NaN,Existing Order,Size,19235,0.155661
2,Payment,6618.0,0.053557,NaN,Returns & Refunds,Not defined by Bot,17245,0.139557
3,Not defined by Bot,4833.0,0.039111,NaN,Returns & Refunds,Return status,12934,0.104669
4,Apps & Website,2071.0,0.016760,NaN,Existing Order,Not defined by Bot,10052,0.081347


In [15]:
df_bot.info()

<class 'pandas.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Contact Reason     15 non-null     str    
 1   Conteo de Filas    15 non-null     float64
 2   Percentage         15 non-null     float64
 3   Unnamed: 3         0 non-null      float64
 4   Contact Reason.1   77 non-null     str    
 5   Sub Category       77 non-null     str    
 6   Conteo de Filas.1  77 non-null     int64  
 7   Percentage.1       77 non-null     float64
dtypes: float64(4), int64(1), str(3)
memory usage: 4.9 KB


### Interpretación
**(77, 8)**

Muy pequeño.

**Esto sugiere firmemente que:**
El bot totalmente automatizado gestiona con éxito un **conjunto relativamente limitado de intenciones**.

> **Esto ya representa una importante información para el negocio.**

# Section 6 — Missing Values

In [16]:
df_agent.isna().sum()

Contact Reason       168
Conteo de Filas      166
Percentage           166
Unnamed: 3           184
Contact Reason.1       0
Sub Category           0
Conteo de Filas.1      0
Percentage.1           0
dtype: int64

In [17]:
df_hybrid.isna().sum()

Contact Reason       189
Conteo de Filas      189
Percentage           189
Unnamed: 3           208
Contact Reason.1       0
Sub Category           0
Conteo de Filas.1      0
Percentage.1           0
dtype: int64

In [18]:
df_bot.isna().sum()

Contact Reason       62
Conteo de Filas      62
Percentage           62
Unnamed: 3           77
Contact Reason.1      0
Sub Category          0
Conteo de Filas.1     0
Percentage.1          0
dtype: int64

## El descubrimiento más importante hasta ahora

> **El conjunto de datos está dividido estructuralmente en dos tablas dentro de cada hoja.**

### LADO IZQUIERDO
- Contact Reason
- Conteo de Filas
- Percentage

Esto es:
**HIGH-LEVEL CATEGORY AGGREGATION/ AGREGACIÓN DE CATEGORÍAS DE ALTO NIVEL** porque son categorías* MUY generales*, *no describen una acción específica* sino *dominios de negocio* y se *están sumando TODAS las conversaciones* pertenecientes a esa macro categoría.

Ejemplo:
- Returns & Refunds (Devoluciones y reembolsos): no es específico. Pueden existir `return status`, `refund pending`, `label request`, `damaged item`
- Existing Order (Orden existente): tampoco es específico. Puede incluir `tracking`, `delivery`, `in transit`, `modify address`
- Payment (Pago): 

Estas categorías funcionan como ***Categorías padre / macro categorías***

```
Returns & Refunds = return status + refund pending + label request + damaged item
```


### LADO DERECHO
- Contact Reason.1
- Sub Category
- Conteo de Filas.1
- Percentage.1

Esto es:
**SUBCATEGORY BREAKDOWN/DESGLOSE DE SUBCATEGORÍA***

Ejemplo:
- Return status
- Size
- Explain how to order

La propia estructura del Excel ya nos da la jerarquía.
| Contact Reason.1  | Sub Category  |
| ----------------- | ------------- |
| Returns & Refunds | Return status |
| Existing Order   | Size         |



### Por eso Hay Datos NULOS/NULLS

- Esto NO es **“datos erróneos/dirty data”**.
- Es un **error de formato de la hoja de cálculo**.
> Esto es sumamente importante.

**Ejemplo:**
| Contact Reason | Count/Conteo de Filas |
| :--- | :--- |
| Returns & Refunds | 116308 |

**A continuación, la hoja de cálculo contiene por separado:**
| Contact Reason.1 | Sub Category |
| :--- | :--- |
| Returns & Refunds | Return status |

**Por lo tanto, el archivo de Excel contiene:**
Dos tablas dinámicas independientes combinadas horizontalmente.

> **Esto explica la presencia de valores nulo en las columnas de la tabla de la izquierda..**
```
Contact Reason       168
Conteo de Filas      166

168 nulls
166 nulls
```

## La Columna "Unnamed: 3"
```
 3   Unnamed: 3         0 non-null      float64
```

### Interpretación

Casi con toda seguridad se trata de:
- una columna separadora visual en Excel
- Probablemente insertada para crear espacio entre las dos tablas dinámicas.

**Esta columna debe eliminarse inmediatamente.**

Esto es debido a que la tabla izquierda tiene menos filas reales (16 categorías.).

Ejemplo:

- Contact Reason
- Returns
- Orders
- Payment


Pero la tabla derecha tiene MUCHAS subcategorías. Entonces se rellena:

- las primeras filas con la tabla izquierda,
- y debajo quedan NaN.

**No es data faltante operacional. Es estructura visual del la hoja de Excel.**


### Hallazgos Operacionales Más Importantes

Ahora interpretemos los resultados reales del negocio.

#### A. Gestión exclusivamente por agentes/Agent-Only: Predominan las devoluciones y los pedidos existentes (Returns & Existing Orders)
-   **Returns & Refunds**
-   **Existing Order**

**Esto sugiere:**
La atención a transacciones complejas depende en gran medida de la *intervención humana*.

**Esto suele indicar:**
- Falta de integraciones con el sistema backend.
- Orquestación insuficiente del flujo de trabajo.
- Automatización deficiente de las transacciones.

> **Esto es estratégicamente importante.**

---

#### B. Gestión híbrida/Hybrid: También predominan los pedidos existentes/Existing Orders
**Muy importante.**

**Esto implica:**
1. El bot intenta ejecutar estos flujos de trabajo,
2. pero a menudo no puede completarlos.

> **Esto es una fuga de escalada conversacional.**

---

#### C. Intenciones más fuertes solo para Bot-Only
-   **How to return**
-   **Size**

Estas son:
- **Deterministas:** La respuesta sigue reglas fijas y repetibles.
- **Informativas:** Porque el usuario busca información, no busca ejecutar una transacción compleja. (“How do I return a product?” vs “My refund never arrived and my payment failed”)
- **De bajo riesgo:** si el bot responde incorrectamente, el impacto operacional suele ser menor. (“How to return” es manejable, pero un Error en pago/refund es más crítico; dinero, fraude, compliance, experiencia negativa severa.)
- **Repetibles:** porque miles de usuarios hacen exactamente la misma pregunta (“How to return”, “Return status”, “Size”), por ende son patrones altamente repetitivos e ideales para automatización.

> **Exactamente los tipos de intenciones que los LLM/chatbots manejan mejor.**

---

#### D. "No definido por el bot/ Not defined by Bot"
**Este es uno de los hallazgos más importantes en todo el conjunto de datos.**

**Ejemplo:**
- `Returns & Refunds` → **Not defined by Bot**

**Esto indica claramente:**
- Fallo en el reconocimiento de la intención/intent recognition failure.
- o limitaciones de taxonomía/taxonomy limitations.
- o brechas en el enrutamiento alternativo/fallback routing gaps.

> **Potencialmente las tres.**

## **Conclusión Section 6**:

- Ahora sabemos que los valores nulos, **no son** valores nulos operativos.
- Mas bien son **artifacts** del formato del panel de control de Excel.
- La estructura del Dataset:
    * Entendemos que cada hoja contiene:

**TABLE A — Category Summary**
| contact_reason | count | percentage |
| :--- | :--- | :--- |

<br>

**TABLE B — Subcategory Summary**
| contact_reason | sub_category | count | percentage |
| :--- | :--- | :--- | :--- |



*Dentro de la misma hoja de cálculo.*

---

### Esto implica cambios en nuestro siguiente paso

- No debemos analizar estas hojas directamente.
- **En su lugar:**
    * Debemos extraer y normalizar las tablas por separado.


---

### Lo más importante hasta ahora

> El conjunto de datos ya constituye una **capa de agregación de KPI operativos**, no datos de interacción sin procesar.

Es decir: estamos analizando:
- **Rendimiento del negocio/Business performance**
- **Cuellos de botella operativos/Operational bottlenecks**
- **Eficacia de la automatización/Automation effectiveness**

#### **NO:**
- Conversaciones individuales
- Incrustaciones de PLN
- Análisis de sentimientos

> **Esto define fundamentalmente el alcance de la solución.**

---

Nuestra recomendación final probablemente debería centrarse en:
| Area | Recommendation |
| :--- | :--- |
| **Intent classification** | Improve taxonomy + fallback handling |
| **Transactional flows** | Add Order Management System (OMS)/payment integrations |
| **Hybrid leakage** | Improve escalation decisioning |
| **FAQ automation** | Expand deterministic automation |
| **Analytics instrumentation** | Fix undefined intents |

# Section 7 — Standardize Column Names

In [19]:
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[^\w]", "", regex=True)
    )
    return df

In [20]:
df_agent_clean = clean_columns(df_agent)

In [21]:
df_hybrid_clean = clean_columns(df_hybrid)

In [22]:
df_bot_clean = clean_columns(df_bot)

# Section 8 — Normalize Tables

Transformaremos las Tablas de Reporte de Excel en un Dataset analíticos limpio

Crearemos 4 datasets principales:

| Dataset                 | Nivel            |
| ----------------------- | ---------------- |
| `agent_category_df`     | categorías macro |
| `agent_subcategory_df`  | subcategorías    |
| `hybrid_category_df`    | categorías macro |
| `hybrid_subcategory_df` | subcategorías    |

Y luego consolidaremos todos

## Resultado final esperado

Terminaremos con datasets así:

### Category Dataset
| handling_channel | contact_reason    | volume | percentage |
| ---------------- | ----------------- | -----: | ---------: |
| agent_only       | Returns & Refunds | 116308 |      0.292 |


### Subcategory Dataset
| handling_channel | contact_reason | sub_category | volume | percentage |
| ---------------- | -------------- | ------------ | -----: | ---------: |
| bot_only         | Existing Order | Size         |  19235 |      0.156 |

**¿Por qué esto es MUY importante?**

Porque actualmente el Excel está optimizado para:
- visualización humana,
- reporting manual.

NO para:
- joins,
- dashboards,
- KPI analytics,
- Power BI,
- slicing/filtering,
- modeling.


La Section 8 tendrá esta estructura:

| Paso | Objetivo                         |
| ---- | -------------------------------- |
| 8.1  | Crear función para categorías    |
| 8.2  | Crear función para subcategorías |
| 8.3  | Normalizar Agent                 |
| 8.4  | Normalizar Hybrid                |
| 8.5  | Normalizar Bot                   |
| 8.6  | Consolidar datasets              |
| 8.7  | Validar integridad               |


## SECTION 8.1 — Create Category Extraction Function

In [23]:
def build_category_table(
    df: pd.DataFrame,
    handling_channel: str,
) -> pd.DataFrame:
    """
    Build normalized category-level dataset.

    Parameters
    ----------
    df : pd.DataFrame
        Raw worksheet dataframe.
    handling_channel : str
        Operational handling channel.

    Returns
    -------
    pd.DataFrame
        Clean category-level analytical dataset.
    """

    # Extrae SOLO la tabla izquierda porque esa es la **tabla macro categórica**.
    category_df = (
        df[
            [
                "contact_reason",
                "conteo_de_filas",
                "percentage",
            ]
        ]
        .dropna(subset=["contact_reason"]) # Elimina filas vacías
        .rename(
            columns={
                "conteo_de_filas": "volume", # Renombramos columnas: nombres más consistentes y limpios
            }
        )
        .copy()
    )

    # Agregamos handling_channel para comparar agent vs hybrid vs bot
    category_df["handling_channel"] = handling_channel

    category_df = category_df[
        [
            "handling_channel",
            "contact_reason",
            "volume",
            "percentage",
        ]
    ]

    return category_df

## SECTION 8.2 — Create Subcategory Function

In [24]:
def build_subcategory_table(
    df: pd.DataFrame,
    handling_channel: str,
) -> pd.DataFrame:
    """
    Build normalized subcategory-level dataset.

    Parameters
    ----------
    df : pd.DataFrame
        Raw worksheet dataframe.
    handling_channel : str
        Operational handling channel.

    Returns
    -------
    pd.DataFrame
        Clean subcategory-level analytical dataset.
    """

    # Aquí extraemos la tabla derecha.
    subcategory_df = (
        df[
            [
                "contact_reason1",
                "sub_category",
                "conteo_de_filas1",
                "percentage1",
            ]
        ]
        .rename(
            columns={
                "contact_reason1": "contact_reason",
                "conteo_de_filas1": "volume",
                "percentage1": "percentage",
            }
        )
        .copy()
    )

    subcategory_df["handling_channel"] = handling_channel

    subcategory_df = subcategory_df[
        [
            "handling_channel",
            "contact_reason",
            "sub_category",
            "volume",
            "percentage",
        ]
    ]

    return subcategory_df

## SECTION 8.3 — Build Agent Datasets

Construimos ambos datasets de Agent por categoría y subcategoría

**¿Por qué renombramos contact_reason1?**

Porque después de separar las tablas, ya NO necesitamos:
- .1
- columnas duplicadas.

Ahora **son datasets independientes**.

In [25]:
agent_category_df = build_category_table(
    df=df_agent_clean,
    handling_channel="agent_only",
)

agent_subcategory_df = build_subcategory_table(
    df=df_agent_clean,
    handling_channel="agent_only",
)

## SECTION 8.4 — Build Hybrid Datasets

In [26]:
hybrid_category_df = build_category_table(
    df=df_hybrid_clean,
    handling_channel="hybrid",
)

hybrid_subcategory_df = build_subcategory_table(
    df=df_hybrid_clean,
    handling_channel="hybrid",
)

## SECTION 8.5 — Build Bot Datasets

In [27]:
bot_category_df = build_category_table(
    df=df_bot_clean,
    handling_channel="bot_only",
)

bot_subcategory_df = build_subcategory_table(
    df=df_bot_clean,
    handling_channel="bot_only",
)

## SECTION 8.6 — Consolidate Datasets

Hacemos Merges de los datasets de categorías y subcategorías.

Con esto, ya tenemos un solo Dataset Analítico pues:

**Antes:**
- 3 hojas Excel separadas.

**Ahora:**
- 1 modelo analítico consistente.

### Category Dataset

In [28]:
categories_df = pd.concat(
    [
        agent_category_df,
        hybrid_category_df,
        bot_category_df,
    ],
    ignore_index=True,
)

### Subcategory Dataset

In [29]:
subcategories_df = pd.concat(
    [
        agent_subcategory_df,
        hybrid_subcategory_df,
        bot_subcategory_df,
    ],
    ignore_index=True,
)

## SECTION 8.7 — Validate Final Datasets

In [30]:
print(categories_df.shape)
print(subcategories_df.shape)

(50, 4)
(469, 5)


In [31]:
categories_df

,handling_channel,contact_reason,volume,percentage
0,agent_only,Returns & Refunds,116308.0,0.292082
1,agent_only,Existing Order,106339.0,0.267047
2,agent_only,Customer Feedback,71907.0,0.180579
3,agent_only,Support on Ordering,52426.0,0.131656
4,agent_only,Spam/No Contact,16204.0,0.040693
5,agent_only,Payment,12257.0,0.030781
6,agent_only,Membership,6712.0,0.016856
7,agent_only,Defective Returns Management,4494.0,0.011286
8,agent_only,Vouchers & Gift cards,3376.0,0.008478
9,agent_only,Product Information,2926.0,0.007348


In [32]:
subcategories_df

,handling_channel,contact_reason,sub_category,volume,percentage
0,agent_only,Customer Feedback,Left Blank,70731,0.177625
1,agent_only,Support on Ordering,Explain how to order,52148,0.130958
2,agent_only,Returns & Refunds,Left Blank,36494,0.091647
3,agent_only,Returns & Refunds,Return status,30817,0.077390
4,agent_only,Existing Order,Size,20702,0.051989
...,...,...,...,...,...
464,bot_only,Customer Feedback,Left Blank,1,0.000008
465,bot_only,Existing Order,Duplicate order,1,0.000008
466,bot_only,Membership,Cannot claim/ use reward,1,0.000008
467,bot_only,Membership,Voucher not working,1,0.000008


# Duplicados

- no hubo duplicación accidental
- no hubo concatenaciones erróneas
- no hay registros repetidos.

In [33]:
categories_df.duplicated().sum()

np.int64(0)

In [34]:
subcategories_df.duplicated().sum()

np.int64(0)

# Percentages

- no se perdieron filas
- no se duplicaron categorías
- las agregaciones originales siguen consistentes.

In [35]:
categories_df.groupby(
    "handling_channel"
)["percentage"].sum()

handling_channel
agent_only    1.0
bot_only      1.0
hybrid        1.0
Name: percentage, dtype: float64

# Data types

In [36]:
categories_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   handling_channel  50 non-null     str    
 1   contact_reason    50 non-null     str    
 2   volume            50 non-null     float64
 3   percentage        50 non-null     float64
dtypes: float64(2), str(2)
memory usage: 1.7 KB


In [37]:
subcategories_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 469 entries, 0 to 468
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   handling_channel  469 non-null    str    
 1   contact_reason    469 non-null    str    
 2   sub_category      469 non-null    str    
 3   volume            469 non-null    int64  
 4   percentage        469 non-null    float64
dtypes: float64(1), int64(1), str(3)
memory usage: 18.4 KB


# Section 9 — KPI Engineering & Operational Metrics

## Objetivo de la Section 9

Construir:

- métricas operacionales,
- KPIs de automatización,
- indicadores de fricción,
- oportunidades de mejora.

Aquí ya no solo limpiamos datos:
1. interpretamos operación,
2. detectamos oportunidades,
3. construimos narrativa ejecutiva.

### **IMPORTANTE**

Como NO tenemos:

- timestamps
- sessions
- resolution outcomes
- CSAT
- conversation logs

debemos construir:
`proxy KPIs`

¿Qué construiremos?

| KPI                    | Tipo           |
| ---------------------- | -------------- |
| Channel distribution   | Operacional    |
| Top intents            | CX             |
| Escalation leakage     | Conversational |
| Undefined intent rate  | NLP / taxonomy |
| Automation opportunity | Estratégico    |
| Pareto concentration   | Priorización   |
| Intent handling matrix | Operacional    |


---

### Estructura de la Section 9
| Paso | Objetivo                        |
| ---- | ------------------------------- |
| 9.1  | Total contact volume            |
| 9.2  | Channel distribution            |
| 9.3  | Top intents overall             |
| 9.4  | Top intents by channel          |
| 9.5  | Automation opportunity analysis |
| 9.6  | Undefined intent analysis       |
| 9.7  | Pareto analysis                 |
| 9.8  | Strategic findings              |


## SECTION 9.1 — Total Contact Volume

Esto representa:
- **total operational contact volume captured in the reporting period** (Volumen total de contactos operativos registrados en el período del informe)

Este número:

- dimensiona el problema,
- justifica automatización,
- cuantifica impacto potencial.

In [38]:
total_contacts = categories_df["volume"].sum()

print(f"Total contacts: {total_contacts:,.0f}")

Total contacts: 798,617


Estamos hablando de: $~800 mil$ contactos operacionales

Eso inmediatamente:
- justifica inversión en automatización,
- justifica optimización del bot,
- justifica analytics conversacional.

Ya que si se automatizan al menos el $5%$, $10%$ o $15%$ de esos contactos, el el impacto operacional es signiticativo. Por ejemplo:

Si reducimos en $80,000$ contactos, eso puede representar:

- menos costo operativo
- menor tiempo de espera
- mejor SLA (Service Level Agreement), es decir, mejor nivel de calidad y rendimiento que se espera de un servicio específico
- menos carga de agentes.

## SECTION 9.2 — Channel Distribution

cómo se distribuye el volumen
entre:
- humanos,
- híbrido,
- bot

In [39]:
channel_distribution = (
    categories_df
    .groupby("handling_channel")["volume"]
    .sum()
    .reset_index()
    .sort_values(by="volume", ascending=False)
)

channel_percentages = channel_distribution.copy()
# Calcular el porcentaje sobre el total del volumen
channel_percentages["percentage"] = (
    channel_percentages["volume"] / channel_percentages["volume"].sum()
) * 100
channel_percentages.round(1)

,handling_channel,volume,percentage
0,agent_only,398203.0,49.9
2,hybrid,276844.0,34.7
1,bot_only,123570.0,15.5


### Interpretación operacional

- **Agent Only domina:** gran parte de la operación todavía depende completamente de humanos
- **Bot-only es el más pequeño:** eso es crítico porque el bot sí participa, pero contiene completamente relativamente poco, es decir, desvía volumen operativo pero registra una baja tasa de resolución definitiva en el primer contacto.
- **Hybrid es enorme:** Y aquí está probablemente el insight MÁS importante del proyecto, pues al ser una combinación de `bot + humano`, el bot está participando, PERO no está resolviendo completamente. Esto es EXACTAMENTE lo que llamamos ***Conversational Escalation Leakage***

**¿Por qué “leakage”?**

Porque la conversación entra al bot pero **“se fuga”** hacia agentes.

Operacionalmente significa:

| Problema posible        | Significado           |
| ----------------------- | --------------------- |
| Mala clasificación      | NLP insuficiente      |
| Flujos incompletos      | journeys rotos        |
| Integraciones faltantes | no acceso OMS         |
| UX conversacional débil | usuarios abandonan    |
| Baja confianza          | usuarios piden humano |

#### Conlusión preliminar:
> El chatbot parece participar de forma significativa en la experiencia del cliente, pero su capacidad de contención sigue siendo limitada, como lo demuestra el gran volumen de consultas gestionadas de forma híbrida.

## SECTION 9.3 — Top Intents Overall

Aquí buscamos identificar drivers principales de contacto

**Insight esperado**

Dominarán casi seguro :
- Returns & Refunds
- Existing Order

Eso es MUY típico en retail e-commerce.

In [40]:
overall_intents = (
    categories_df
    .groupby("contact_reason")["volume"]
    .sum()
    .reset_index()
    .sort_values(by="volume", ascending=False)
)

overall_intents.head(10)

,contact_reason,volume
4,Existing Order,270531.0
12,Returns & Refunds,245084.0
2,Customer Feedback,78932.0
15,Support on Ordering,58100.0
14,Spam/No Contact,44314.0
9,Payment,32024.0
5,Membership,14998.0
11,Product Information,14785.0
16,Vouchers & Gift cards,9899.0
3,Defective Returns Management,8421.0


| Intent              | Volume |
| ------------------- | -----: |
| Existing Order      |   270k |
| Returns & Refunds   |   245k |
| Customer Feedback   |    78k |
| Support on Ordering |    58k |

Los dos dominantes son: 
1. Existing Order
2. Returns & Refunds

Porque son:
- post-purchase interactions
- altamente frecuentes
- operacionalmente repetitivos

**Lo MÁS importante**
Muchos de estos intents son **altamente automarizables/highly automatable**

### Existing Order
Incluye:
- tracking
- shipping status
- order visibility
- delays

### Returns & Refunds
Incluye:
- return labels,
- return status,
- refund tracking.

Estas categorías *NO requieren razonamiento complejo humano*, sino de:
- acceso a sistemas
- lógica determinística
- APIs
- orchestration

**Conclusión premilinar:**
Esto conecta PERFECTAMENTE con LAM / AI Agent vision porque:

- los LAMs funcionan MUY bien cuando hay `workflows`, `herramientas`,`APIs`, `estado transaccional`.


## SECTION 9.4 — Top Intents by Channel

Aquí ya comenzamos a ver el `intent-channel behavior`

**Por ejemplo:**

Si:

| Intent         | Agent | Hybrid   | Bot  |
| -------------- | ----- | -------- | ---- |
| Existing Order | alto  | MUY alto | alto |


Entonces:
- el bot entiende parcialmente,
- pero necesita escalamiento.

In [41]:
top_intents_by_channel = (
    categories_df
    .pivot_table(
        index="contact_reason",
        columns="handling_channel",
        values="volume",
        aggfunc="sum",
    )
    .fillna(0)
)

# Ordenar el índice basándonos en la suma horizontal calculada al vuelo
top_intents_by_channel = top_intents_by_channel.loc[
    top_intents_by_channel.sum(axis=1).sort_values(ascending=False).index
]

top_intents_by_channel

handling_channel,agent_only,bot_only,hybrid
contact_reason,,,
Existing Order,106339.0,50294.0,113898.0
Returns & Refunds,116308.0,54282.0,74494.0
Customer Feedback,71907.0,1.0,7024.0
Support on Ordering,52426.0,617.0,5057.0
Spam/No Contact,16204.0,1322.0,26788.0
Payment,12257.0,6618.0,13149.0
Membership,6712.0,419.0,7867.0
Product Information,2926.0,823.0,11036.0
Vouchers & Gift cards,3376.0,784.0,5739.0


### 1. Existing Order
| Channel | Volume |
| ------- | -----: |
| Agent   |   106k |
| Hybrid  |   113k |
| Bot     |    50k |

Lo anterior, significa que el bot:
- sí entra,
- sí participa,
- sí resuelve parcialmente.

PERO la mayoría termina escalando Porque: 
```
Hybrid > Bot-only
```

Eso sugiere:
El **intent se reconoce** PERO **no se completa**.

**Hipótesis**
| Hipótesis                | Significado                  |
| ------------------------ | ---------------------------- |
| Falta integración OMS    | bot no consulta pedidos      |
| Fallo de autenticación   | no puede verificar usuario   |
| Journey incompleto       | flujo termina prematuramente |
| Usuario pierde confianza | pide humano                  |

**Ejemplo**

1. Cuando se pregunta:

```
“Where is my order?”
```

el bot necesita:
- consultar OMS
- recuperar estado
- responder dinámicamente.

2. Si NO tiene integración OMS:

el bot:
- reconoce el intent, PERO no puede resolver.

Y entonces:
- escala a humano.




### 2. Customer Feedback

| Channel | Volume |
| ------- | -----: |
| Agent   |    71k |
| Hybrid  |     7k |
| Bot     |      1 |

Esto puede significar que el sistema aparentemente casi nunca automatiza feedback, porque:

- requiere empatía
- puede involucrar frustración
- puede ser ambiguo
- puede requerir judgment humano

Esto **NO necesariamente es malo**. De hecho, **puede ser decisión operacional deliberada**.



### 3. Support on Ordering

| Channel | Volume |
| ------- | -----: |
| Agent   |    52k |
| Hybrid  |     5k |
| Bot     |    617 |

Porque `Explain how to order` debería ser:
- FAQ simple
- altamente automatizable

Pero NO lo está siendo y eso puede indicar:
| Posible problema       | Interpretación           |
| ---------------------- | ------------------------ |
| Poor conversational UX | usuarios no entienden    |
| Mala discoverability   | bot no guía bien         |
| Knowledge gaps         | respuestas insuficientes |
| Intent routing pobre   | clasificación mala       |


## SECTION 9.5 — Automation Opportunity Analysis

Aquí comienza el verdadero valor estratégico, en donde estamos identificando: `high-volume repetitive intents`

**¿Por qué estos intents?**

Porque suelen ser:

- determinísticos,
- FAQ-like,
- workflow-driven,
- integrables con APIs.

Con esto, podemos ***encontrar las intenciones que representan un alto potencial de automatización***.

In [42]:
automation_candidates = subcategories_df[
    subcategories_df["sub_category"].isin(
        [
            "Return status",
            "How to return",
            "Explain how to order",
            "Size",
            "label Request",
            "Payment Process Information",
            "In transit",
        ]
    )
]

automation_candidates

,handling_channel,contact_reason,sub_category,volume,percentage
1,agent_only,Support on Ordering,Explain how to order,52148,0.130958
3,agent_only,Returns & Refunds,Return status,30817,0.077390
4,agent_only,Existing Order,Size,20702,0.051989
7,agent_only,Returns & Refunds,label Request,12139,0.030484
8,agent_only,Existing Order,In transit,11238,0.028222
11,agent_only,Returns & Refunds,How to return,6856,0.017217
38,agent_only,Payment,Payment Process Information,1275,0.003202
131,agent_only,Existing Order,Explain how to order,4,0.000010
139,agent_only,Returns & Refunds,Explain how to order,3,0.000008
142,agent_only,Customer Feedback,Return status,2,0.000005


### Dónde invertir automatización

| Intent               | Volume |
| -------------------- | -----: |
| Explain how to order |    52k |
| Return status        |    30k |
| Size                 |    20k |
| label Request        |    12k |
| In transit           |    11k |


Porque son intents:

| Característica  | Sí |
| --------------- | -- |
| Repetitivos     | ✅  |
| Determinísticos | ✅  |
| Workflow-based  | ✅  |
| API-driven      | ✅  |
| Alta frecuencia | ✅  |
| Baja ambigüedad | ✅  |

Es decir: 
```
mismo input → mismo tipo de resolución
```

**Ejemplo**
```
“Where is my order?”
```

La resolución:
- consultar tracking,
- devolver estado.

No requiere:
- creatividad,
- razonamiento complejo,
- juicio humano sofisticado.

---

Esto demuestra algo MUY importante pues el bot **NO está fallando completamente**, de hecho **ya existe evidencia clara de automatización exitosa parcial**

Lo que significa que:
- no hay que reconstruir todo
- hay capacidades existentes
- se puede optimizar incrementalmente.

### Conclusión Preliminar
No parece un problema de:

```
"el bot no puede automatizar"
```

Parece más un problema de:
```
“el bot no puede contener completamente flujos de trabajo transaccionales específicos”
```

Esa es una diferencia MUY importante.

## Diagnóstico Preliminar

#### 1. Participación Significativa del Bot
- El chatbot participa activamente en una proporción relevante de las interacciones operacionales, evidenciando una adopción importante dentro del journey de atención al cliente.
- **Sin embargo:** La Tasa de Contención (*Containment Rate*) todavía es limitada para ciertos flujos transaccionales.
- Esto indica que el bot frecuentemente logra iniciar o clasificar conversaciones, pero no siempre completar exitosamente la resolución end-to-end sin intervención humana

#### 2. Dominio Operacional de Procesos Críticos
- Las macrocategorías de **Existing Order** y **Returns & Refunds** concentran la mayor parte del volumen operacional total.
- Esto sugiere que la operación de soporte está fuertemente impulsada por procesos post-compra y workflows transaccionales de alta recurrencia.
- **Oportunidad:** Representan los núcleos con mayor potencial y retorno de inversión para la automatización.

#### 3. Evidencia de Automatización Parcial Exitosa
Se identifican subcategorías donde la automatización autónoma funciona correctamente, concentrada en:
- `Returns` (Consultas informativas de devolución)
- `Size` (Guías y tablas de tallaje)
- `Tracking` (Monitoreo básico de envíos)

debido a que son:
- repetitivas,
- determinísticas,
- orientadas a workflows,
- y potencialmente integrables mediante APIs o sistemas transaccionales.

#### 4. Arquitectura de Falla Identificada
El estancamiento de las métricas no se debe a una sola causa, sino a una combinación de factores estructurales:
* **Escalamiento transaccional:** Fuga de contactos hacia canales humanos a mitad del ciclo de atención.
* **Journeys incompletos:** Experiencias de usuario fragmentadas que no cierran el caso en el primer contacto.
* **Limitaciones de integración:** Falta de conexiones directas con los sistemas core.
* **Clasificación insuficiente:** Brechas de NLP e intents etiquetados como no definidos.

#### 5. Máximo Potencial Estratégico
Para transformar el rendimiento del bot y alcanzar los objetivos del negocio, los esfuerzos de ingeniería deben priorizar de forma inmediata:
* **Workflows transaccionales:** Rediseño de flujos de extremo a extremo sin salir de la interfaz.
* **OMS Integration:** Conexión nativa con el *Order Management System* para automatizar estados de pedidos.
* **Return Orchestration:** Automatización completa de la logística inversa (generación de guías, reembolsos).
* **Conversational Containment Optimization:** Afinación de las reglas de negocio y enrutamiento automatizado para elevar la retención.

# SECTION 10 — Visualization & Exploratory Analytics

## Objetivo de la Section 10

Construir:
- visualizaciones ejecutivas,
- gráficas interpretables,
- KPIs visuales,
- gráficos listos para presentación.

Lo que construiremos:

| Visual                     | Objetivo                  |
| -------------------------- | ------------------------- |
| Channel Distribution       | distribución operacional  |
| Top Intents                | drivers principales       |
| Stacked Channel Comparison | containment vs escalation |
| Pareto Chart               | priorización              |
| Automation Opportunity     | oportunidades             |
| Undefined Intent KPI       | NLP gaps                  |
| Left Blank Analysis        | observabilidad            |


---

**Estructura de la Section 10**
| Paso | Objetivo                         |
| ---- | -------------------------------- |
| 10.1 | Imports                          |
| 10.2 | Channel Distribution Chart       |
| 10.3 | Top Intents Chart                |
| 10.4 | Stacked Channel Comparison       |
| 10.5 | Pareto Chart                     |
| 10.6 | Automation Opportunities         |
| 10.7 | Undefined Intent Visualization   |
| 10.8 | Strategic Visualization Insights |



## SECTION 10.1 — Visualization Imports

In [43]:
import plotly.express as px
import plotly.graph_objects as go

## SECTION 10.2 — Channel Distribution Chart

In [44]:
fig = px.bar(
    channel_distribution,
    x="handling_channel",
    y="volume",
    color="handling_channel",
    title="Contact Volume by Handling Channel",
    text_auto=".2s",
)

fig.update_layout(
    xaxis_title="Handling Channel",
    yaxis_title="Contact Volume",
    template="plotly_white",
)

fig.show()

### Interpretación — Distribución por Canal de Atención

La visualización confirma tres volúmenes operacionales con implicaciones estratégicas críticas:

| Canal | Volumen | Participación |
| :--- | ---: | :---: |
| `agent_only` | 398,203 | **49.9%** |
| `hybrid` | 276,844 | **34.7%** |
| `bot_only` | 123,570 | **15.5%** |
| **Total** | **798,617** | **100%** |

#### A. El bot participa pero no contiene

El chatbot está presente en el **50.2%** de los contactos totales (canales `hybrid` + `bot_only`). Sin embargo, de todos los contactos donde el bot interviene, únicamente logra resolver de forma autónoma el **30.8%**:

$$\text{Tasa de Contención Interna del Bot} = \frac{123{,}570}{276{,}844 + 123{,}570} = \frac{123{,}570}{400{,}414} \approx 30.8\%$$

Este cálculo expone una brecha crítica entre *participación* y *contención real*: el **69.2%** de los contactos donde el bot interviene termina escalando a un agente humano (canal `hybrid`), generando un costo operativo doble.

#### B. El costo oculto del canal híbrido

El canal `hybrid` (276,844 contactos, 34.7%) es el mayor riesgo operacional del sistema. Cada contacto híbrido consume simultáneamente:
- **Capacidad del bot:** inicia la conversación, intenta clasificar y resolver.
- **Capacidad del agente:** recibe el caso, recontextualiza y resuelve desde cero.
- **Tiempo del cliente:** espera la transferencia y repite su problema.

En términos de costo operativo por unidad, los contactos híbridos son los **más costosos del sistema**, pues duplican el consumo de recursos sin garantizar mejor experiencia.

#### C. Diagnóstico de la Containment Rate declarada

El Business Case reporta una **Containment Rate del 48%** (target: 55%). La distribución de esta muestra es consistente con un sistema que enfrenta tres vectores de falla simultáneos:

| Vector de Falla | Evidencia en los datos |
| :--- | :--- |
| Journeys incompletos | `hybrid` (34.7%) supera a `bot_only` (15.5%) |
| Falta de integraciones transaccionales | Categorías OMS-dependientes dominadas por `agent_only` |
| Baja confianza del usuario | Alta escalación en categorías donde el bot sí participa |

> **Conclusión operacional:** La gráfica no muestra un bot fallido, sino un bot *subutilizado*. La capacidad técnica de automatización existe, pero no está completamente desplegada ni conectada a los sistemas de backend necesarios para la resolución end-to-end.

## SECTION 10.3 — Top Intents Overall

In [45]:
total_volume = overall_intents["volume"].sum()

top_10_percentages = overall_intents[["contact_reason"]].copy()
top_10_percentages["percentage"] = (
    overall_intents["volume"] / total_volume
) * 100


print(top_10_percentages.round(2))
print("\nPorcentaje % Acumulado: ", top_10_percentages["percentage"].sum().round(1))

                  contact_reason  percentage
4                 Existing Order       33.87
12             Returns & Refunds       30.69
2              Customer Feedback        9.88
15           Support on Ordering        7.28
14               Spam/No Contact        5.55
9                        Payment        4.01
5                     Membership        1.88
11           Product Information        1.85
16         Vouchers & Gift cards        1.24
3   Defective Returns Management        1.05
0                 Apps & Website        0.87
7             Not defined by Bot        0.73
1            Company Information        0.53
8                          Other        0.45
13                   Running App        0.08
17               Withdrawal Form        0.02
18                adidas Running        0.01
10     Privacy and Data Handling        0.00
6                     Newsletter        0.00

Porcentaje % Acumulado:  100.0


In [46]:
top_10_intents = overall_intents

fig = px.bar(
    top_10_intents,
    x="contact_reason",
    y="volume",
    color="contact_reason",
    title="Top Contact Reasons",
    text_auto=".2s",
)

fig.update_layout(
    xaxis_title="Contact Reason",
    yaxis_title="Volume",
    template="plotly_white",
    xaxis_tickangle=-30,
    showlegend=False,
)

fig.show()

### Interpretación — Top 10 Razones de Contacto

La distribución de volumen por `contact_reason` revela una concentración extrema en pocas categorías:

| # | Contact Reason | Volumen | % del Total | % Acumulado |
| :---: | :--- | ---: | :---: | :---: |
| 1 | Existing Order | 270,531 | 33.9% | 33.9% |
| 2 | Returns & Refunds | 245,084 | 30.7% | **64.6%** |
| 3 | Customer Feedback | 78,932 | 9.9% | 74.5% |
| 4 | Support on Ordering | 58,100 | 7.3% | **81.7%** |
| 5 | Spam / No Contact | 44,314 | 5.6% | 87.3% |
| 6 | Payment | 32,024 | 4.0% | 91.3% |
| 7 | Membership | 14,998 | 1.9% | 93.2% |
| 8 | Product Information | 14,785 | 1.9% | 95.0% |
| 9 | Vouchers & Gift cards | 9,899 | 1.2% | 96.2% |
| 10 | Defective Returns Mgmt | 8,421 | 1.1% | 97.3% |

#### A. Concentración en procesos post-compra

Las dos categorías dominantes acumulan más del **64.6%** del volumen total de contactos:

$$\text{Top 2} = 270{,}531 + 245{,}084 = 515{,}615 \implies \frac{515{,}615}{798{,}617} = \mathbf{64.6\%}$$

Esto es característico del retail digital: los clientes contactan masivamente en la fase *post-transaccional*. Crucialmente, estas categorías **no requieren razonamiento empático ni juicio complejo**; son *workflow-driven* con resoluciones determinísticas y repetibles: consultar un sistema, devolver un estado, generar una etiqueta.

#### B. Customer Feedback como señal de fricción sistémica

El tercer driver (**Customer Feedback: 9.9%**, 78,932 contactos) debe interpretarse como un indicador indirecto de insatisfacción acumulada del cliente. En un sistema bien calibrado, el feedback genuino debería representar un volumen considerablemente menor. Un volumen tan elevado sugiere que muchos clientes canalizan bajo este intent genérico **frustraciones o problemas no resueltos en interacciones previas**.

#### C. Support on Ordering — La Oportunidad FAQ más inmediata

**Support on Ordering** (7.3%, 58,100 contactos) está dominado por el sub-intent `Explain how to order`. Esta es una consulta puramente informacional que **no requiere ninguna integración sistémica** para automatizarse. La presencia de >50,000 consultas FAQ básicas en canal humano indica una brecha directa de diseño conversacional, con **cero dependencias técnicas externas** para resolverse.

> **Implicación estratégica:** Concentrar los esfuerzos de optimización en las cuatro primeras categorías garantiza impacto directo sobre el **81.7% del volumen operacional total**.

## SECTION 10.4 — Stacked Channel Comparison

In [47]:
stacked_df = (
    categories_df
    .pivot_table(
        index="contact_reason",
        columns="handling_channel",
        values="volume",
        aggfunc="sum",
    )
    .reset_index()
)

fig = px.bar(
    stacked_df,
    x="contact_reason",
    y=["agent_only", "hybrid", "bot_only"],
    title="Intent Distribution by Handling Channel",
)

fig.update_layout(
    xaxis_title="Contact Reason",
    yaxis_title="Volume",
    template="plotly_white",
    xaxis_tickangle=-35,
)

fig.show()

### Interpretación — Distribución por Canal y Categoría (Vista Apilada)

Esta visualización permite identificar el perfil de escalamiento de cada categoría, distinguiendo dónde el bot falla por ausencia de diseño, dónde falla por falta de integración, y dónde ya funciona relativamente bien.

#### A. Patrones de Falla Crítica — Dominancia Absoluta de `agent_only`

| Categoría | Patrón visual | Diagnóstico |
| :--- | :--- | :--- |
| **Customer Feedback** | Barra casi 100% azul | Bot no procesa feedback. Routing directo a agente. |
| **Support on Ordering** | Barra casi 100% azul | 52k+ contactos FAQ básicos resueltos por humanos. |
| **Spam / No Contact** | Alta proporción roja (hybrid) | Ruido operacional con alta participación híbrida. |

Estas categorías exhiben una participación verde (bot_only) prácticamente inexistente. Esto **no responde a limitaciones de NLP**, sino a la ausencia de flujos conversacionales diseñados para ellas: son **brechas de cobertura**, no brechas de inteligencia.

#### B. El Patrón Híbrido — Journeys Incompletos

| Categoría | Patrón visual | Diagnóstico |
| :--- | :--- | :--- |
| **Existing Order** | Barra más alta (~260k). Rojo domina sobre verde | Bot reconoce el intent pero no completa el journey |
| **Returns & Refunds** | Segunda barra (~245k). Alto azul y rojo, algo de verde | Alta fuga hacia agentes en transacciones de devolución |
| **Payment** | Proporciones similares entre los tres canales | Flujos de pago incompletos o sin integración |

En `Existing Order`, el canal `hybrid` visualmente supera al `bot_only`, confirmando el hallazgo central: el bot *identifica* la solicitud pero no puede *ejecutar* la resolución sin acceso a sistemas transaccionales (OMS, carrier APIs).

#### C. Categorías de La Cola Larga

Las categorías de menor volumen (*Newsletter*, *Withdrawal Form*, *adidas Running*, *Privacy and Data Handling*) muestran barras muy pequeñas con comportamientos mixtos. Su bajo volumen las posiciona como prioridad baja —tratables con flujos FAQ simples— una vez consolidadas las categorías principales.

#### D. Validación del Patrón Estructural

La gráfica confirma visualmente la hipótesis central del análisis:

> **Donde no se requiere integración sistémica, el bot funciona relativamente bien.**
> **Donde sí se requiere integración transaccional, el bot falla o escala.**

Esta lectura define con precisión la agenda técnica: el cuello de botella **no es el NLP**, son las integraciones con sistemas core (OMS, plataforma de devoluciones, carrier APIs).


## SECTION 10.5 — Pareto Analysis

Esto nos permite responder `¿Qué pocas categorías generan la mayoría del volumen?` en donde podemos identificar:
- priorización operacional
- priorización de automatización
- priorización de impacto de negocio.

Porque probablemente las categorías de `Existing Order` y `Returns & Refunds` explican más del 60% del volumen total. Lo que significa que si optimizamos esas categorías, impactamos la mayor parte de la operación


In [50]:
pareto_df = overall_intents.sort_values("volume", ascending=False).copy()

pareto_df["cumulative_volume"] = pareto_df["volume"].cumsum()

pareto_df["cumulative_percentage"] = (
    pareto_df["cumulative_volume"]
    / pareto_df["volume"].sum()
)

pareto_df

,contact_reason,volume,cumulative_volume,cumulative_percentage
4,Existing Order,270531.0,270531.0,0.338749
12,Returns & Refunds,245084.0,515615.0,0.645635
2,Customer Feedback,78932.0,594547.0,0.744471
15,Support on Ordering,58100.0,652647.0,0.817222
14,Spam/No Contact,44314.0,696961.0,0.872710
9,Payment,32024.0,728985.0,0.912809
5,Membership,14998.0,743983.0,0.931589
11,Product Information,14785.0,758768.0,0.950102
16,Vouchers & Gift cards,9899.0,768667.0,0.962498
3,Defective Returns Management,8421.0,777088.0,0.973042


In [123]:
fig = go.Figure()

# Bars
fig.add_trace(
    go.Bar(
        x=pareto_df["contact_reason"],
        y=pareto_df["volume"],
        name="Volume",
    )
)

# Pareto line
fig.add_trace(
    go.Scatter(
        x=pareto_df["contact_reason"],
        y=pareto_df["cumulative_percentage"],
        name="Cumulative %",
        yaxis="y2",
        mode="lines+markers",
    )
)

fig.update_layout(
    title="Pareto Analysis of Contact Reasons",
    xaxis_title="Contact Reason",
    yaxis_title="Contact Volume",
    yaxis2=dict(
        title="Cumulative Percentage",
        overlaying="y",
        side="right",
        tickformat=".0%",
    ),
    template="plotly_white",
    xaxis_tickangle=-35,
)

fig.show()

### Interpretación — Análisis de Pareto

Los datos confirman una concentración de contactos que sigue el **Principio de Pareto (Regla 80/20)** con una eficiencia aún más pronunciada sobre las 19 categorías del dataset:

| N° Categorías | Categoría Marginal | Vol. Acumulado | % Acumulado |
| :---: | :--- | ---: | :---: |
| Top 1 | Existing Order | 270,531 | 33.9% |
| Top 2 | Returns & Refunds | 515,615 | **64.6%** |
| Top 3 | Customer Feedback | 594,547 | 74.4% |
| Top 4 | Support on Ordering | 652,647 | **81.7%** |
| Top 6 | Payment | 728,985 | 91.3% |
| Top 12 | Not defined by Bot | 789,884 | 98.9% |
| **Top 19** | Newsletter | **798,617** | **100.0%** |

$$
\text{Alcance de Causas:} \quad \frac{4 \text{ categorías}}{19 \text{ totales}} \approx 21\% \text{ del espectro de intents}
$$

$$
\text{Impacto en Volumen:} \quad \frac{652,647 \text{ contactos}}{798,617 \text{ totales}} \approx 81.7\% \text{ del volumen operativo}
$$

#### A. Implicación Estratégica Directa

> **Optimizar 4 de las 19 categorías impacta el 81.7% del volumen operacional total.**

No se necesita mejorar todos los intents de forma uniforme. Una estrategia **concentrada** en las cuatro categorías dominantes maximiza el retorno por unidad de inversión.

#### B. Validación Cuantitativa del Roadmap 30/60/90

La curva de Pareto justifica directamente la estructura del roadmap propuesto:

| Fase | Categorías Target | Vol. Cubierto | % Acumulado |
| :--- | :--- | ---: | :---: |
| 0–30 días | Support on Ordering + R&R (routing fix) | ~303k | ~38% |
| 30–60 días | Existing Order + R&R (OMS integration) | ~516k | **~65%** |
| 60–90 días | Consolidación top 4 + conversión Hybrid→Bot | ~653k | **~82%** |

#### C. La Cola Larga y el Riesgo Cualitativo de `Not defined by Bot`

Las 15 categorías restantes (~18.3% del volumen) incluyen un caso atípico: `Not defined by Bot` (5,855 contactos, posición 12 en el ranking). Aunque su peso cuantitativo es pequeño, su impacto **cualitativo** es desproporcionado: representa contactos que el sistema no puede ni categorizar, bloqueando cualquier proceso de resolución automatizada. Debe tratarse como **prerequisito técnico**, no como prioridad baja.

> **Conclusión de priorización:** La curva de Pareto convierte la intuición estratégica en imperativo cuantitativo: concentrar, no dispersar. Las cuatro primeras categorías son el campo de batalla donde se gana o pierde la Containment Rate y la Resolution Rate del Business Case.

## SECTION 10.6 — Automation Opportunity Visualization

In [124]:
automation_summary = (
    automation_candidates
    .groupby("sub_category")["volume"]
    .sum()
    .reset_index()
    .sort_values(by="volume", ascending=False)
)

automation_summary

,sub_category,volume
4,Return status,61857
5,Size,60517
0,Explain how to order,56523
1,How to return,38538
2,In transit,29947
6,label Request,28405
3,Payment Process Information,3139


Aquí respondemos a: 
```
¿Qué workflows son mejores candidatos para automatizar?
```

Debido a que estas categorías:
- **NO** requieren razonamiento complejo,
- **NO** son emocionalmente sensibles,
- **NO** requieren juicio humano avanzado.

Sino:
- información/data
- tracking
- lookup
- reglas definidas
- flujos/workflows


In [125]:
fig = px.bar(
    automation_summary,
    x="sub_category",
    y="volume",
    color="sub_category",
    title="Automation Opportunity by Subcategory",
    text_auto=".2s",
)

fig.update_layout(
    xaxis_title="Subcategory",
    yaxis_title="Volume",
    template="plotly_white",
    xaxis_tickangle=-35,
    showlegend=False,
)

fig.show()

### Interpretación — Oportunidades de Automatización por Subcategoría

La gráfica cuantifica el volumen **total cross-channel** de los principales intents candidatos a automatización, incluyendo todos los canales (`agent_only`, `hybrid`, `bot_only`):

| Sub-categoría | Volumen Total | Tipo | Complejidad |
| :--- | ---: | :--- | :---: |
| Return status | **61,857** | Transaccional (OMS) | 🟡 Media |
| Size | **60,517** | FAQ informacional | 🟢 Baja |
| Explain how to order | **56,523** | FAQ informacional | 🟢 Baja |
| How to return | **38,538** | FAQ + Transaccional | 🟡 Media |
| In transit | **29,947** | Transaccional (tracking API) | 🟡 Media |
| label Request | **28,405** | Transaccional (returns system) | 🟡 Media |
| Payment Process Information | **3,139** | FAQ informacional | 🟢 Baja |
| **Total** | **278,926** | | |

La complejidad de automatizar un intent no se basa en qué tan difícil es para la IA entender la pregunta, sino en qué dependencias técnicas (APIs, Bases de datos) necesita el bot para dar una respuesta final.

Así lo clasificamos:

1. 🟢 Complejidad Baja (Informacional / FAQ):
    - Ejemplos: `Size`, `Explain how to order`,` Payment Process Information`.
    - **Por qué es baja:** Porque no requieren integrarse con ningún sistema externo. La respuesta es estática ("Para pedir tu talla, revisa esta tabla..."). El bot solo tiene que reconocer la intención y escupir un texto o un link. Se configuran en horas o días.
2. 🟡 Complejidad Media (Transaccional / Integraciones):
    - Ejemplos: `Return status`, `In transit`, `label Request`.
    - **Por qué es media:** Porque el bot necesita conectarse al backend de la empresa. Si un cliente dice "¿Dónde está mi pedido?", el bot tiene que: 1. Autenticar al usuario, 2. Extraer el número de orden, 3. Hacer una petición API al OMS (Order Management System) o a FedEx/DHL, 4. Procesar el JSON de respuesta y 5. Dárselo al cliente. Requiere desarrollo de ingeniería de software.
3. 🔴 Complejidad Alta (Juicio Humano / Multi-paso complejo):
    - Ejemplos: `Defective Returns Mgmt` o `Customer Feedback`.
    - **Por qué es alta:** Porque involucra excepciones, pedir fotos del producto dañado, evaluar si aplica o no la garantía, y requiere empatía para lidiar con clientes frustrados. Estas se deben escalar a humanos de forma inteligente.

#### A. Automatización Informacional — Quick Wins sin Dependencias Sistémicas

Los intents de **tipo FAQ** (`Size`, `Explain how to order`, `Payment Process Information`) presentan el menor costo de implementación porque:
- No requieren integración con sistemas externos (OMS, carrier APIs, plataforma de devoluciones).
- Sus respuestas son estáticas o semi-estáticas; se sirven desde una base de conocimiento estructurada.
- No tienen dependencias técnicas externas → candidatos para el roadmap de **0–30 días**.

#### B. Automatización Transaccional — Mayor Impacto, Mayor Esfuerzo

Los intents transaccionales (`Return status`, `In transit`, `label Request`) requieren la siguiente cadena de integración:

```
Usuario → Bot (reconoce intent) → Autenticación → API (OMS / Carrier / Returns) → Respuesta dinámica → Resolución confirmada → Cierre del caso
```

El impacto potencial es el mayor del dataset, pero exige **integraciones con sistemas core** → roadmap de **30–60 días**.

#### C. `How to return` — El Intent de Mayor Eficiencia de Mejora

Con 38,538 contactos totales, `How to return` es un caso especial: el flujo **ya existe** en el bot (es una de las subcategorías más robustas del canal `bot_only`), pero sigue presentando un volumen significativo en canales no-bot. Esto indica **fugas de routing**, no ausencia de capacidad. No se necesita construir un flujo desde cero — basta con corregir las reglas de enrutamiento, lo que lo convierte en la oportunidad de **mayor eficiencia de mejora por unidad de esfuerzo**.

#### D. Estimación Conservadora del Impacto en KPIs

Los 278,926 contactos representan el volumen total (incluyendo la porción ya atendida por el bot). Aplicando una tasa de automatización del **60%** sobre la fracción no-bot estimada:

$$\text{Nuevos contactos en bot\_only} \approx 278{,}926 \times 0.40_{\text{non-bot}} \times 0.60_{\text{automation rate}} \approx \mathbf{67{,}000}$$

Estos ~67,000 contactos adicionales en `bot_only` representarían un incremento en la **Containment Rate** del orden de **+5 a +7pp**, suficiente para cerrar la brecha del Business Case (target: 55%).

> **Conclusión de oportunidad:** El potencial está concentrado en 3–4 workflows de alta frecuencia y baja ambigüedad. No se necesita reconstruir el bot; se necesita extender su cobertura sobre intents que ya domina conceptualmente pero no está terminando de resolver en la práctica.


## SECTION 10.7 — Undefined Intent Analysis

Aquí mostramos:

- limitaciones NLP
- gaps taxonómicos
- problemas de clasificación.

In [127]:
undefined_df = categories_df[
    categories_df["contact_reason"]
    == "Not defined by Bot"
]

undefined_df

,handling_channel,contact_reason,volume,percentage
29,hybrid,Not defined by Bot,1022.0,0.003692
38,bot_only,Not defined by Bot,4833.0,0.039111


In [128]:
fig = px.bar(
    undefined_df,
    x="handling_channel",
    y="volume",
    color="handling_channel",
    title="Undefined Intent Volume",
    text_auto=".2s",
)

fig.update_layout(
    xaxis_title="Handling Channel",
    yaxis_title="Volume",
    template="plotly_white",
)

fig.show()

### Interpretación — Volumen de Intents No Definidos por el Bot

La visualización muestra el volumen de contactos clasificados a nivel de **categoría** como `Not defined by Bot`, es decir, contactos que el sistema no pudo asignar a ninguna categoría del árbol de intents:

| Canal | Volumen | % del Total "Not Defined" | Interpretación |
| :--- | ---: | :---: | :--- |
| `bot_only` | **4,833** | **82.5%** | El bot contuvo el contacto **sin poder categorizarlo** |
| `hybrid` | **1,022** | **17.5%** | El bot no pudo clasificar → escaló a agente |
| `agent_only` | **0** | **0%** | Los agentes siempre logran clasificar manualmente |
| **Total** | **5,855** | **100%** | 0.73% del volumen total |

#### A. El Hallazgo Más Preocupante: Bot-only con Intent No Definido

La distribución revela un patrón contraintuitivo y crítico: el **82.5% del volumen no definido** pertenece al canal `bot_only`. Esto significa que el bot está **conteniendo** 4,833 contactos que no puede categorizar, en lugar de escalarlos para que un agente los resuelva correctamente.

Desde la perspectiva de la **Resolution Rate**, este es el peor escenario posible:
- El contacto se registra como *contenido* → infla artificialmente la Containment Rate.
- El cliente recibió una respuesta genérica de fallback → **problema no resuelto**.
- El cliente volverá a contactar → infla la Repeat Rate.

$$\text{Efecto en KPIs:} \quad \text{Falsa Contención} \rightarrow \text{Sin Resolución} \rightarrow \text{Repeat Rate} \uparrow$$

#### B. Diagnóstico de Causas Raíz

| Causa | Descripción | Remediación |
| :--- | :--- | :--- |
| **NLU Coverage Gap** | El modelo no tiene entrenamiento para estos patrones de lenguaje | Reentrenamiento con nuevas utterances etiquetadas |
| **Taxonomy Gap** | El árbol de intents no cubre todas las categorías de negocio existentes | Expansión de la taxonomía de categorías |
| **Fallback Routing** | El bot no transfiere al agente cuando no puede clasificar | Diseño de escalamiento graceful con recolección de contexto |

#### C. Por qué `agent_only = 0` es un Dato Relevante

La ausencia total del canal `agent_only` en esta categoría confirma que los **agentes humanos siempre logran asignar un intent** cuando intervienen. Esto descarta un problema de taxonomía global (las categorías sí existen y son suficientes), y reenfoca el diagnóstico en una brecha específica de **cobertura de entrenamiento NLP** o de **lógica de fallback routing** del bot.

> **Conclusión crítica:** Los 5,855 contactos `Not defined by Bot` representan *falsa contención*: el sistema los cuenta como retenidos pero probablemente no los resuelve. Su impacto cualitativo es desproporcionado respecto a su volumen. Resolverlos es un **prerequisito técnico**, no una optimización opcional.

## SECTION 10.8 — Left Blank Operational Analysis

Ahora analizaremos:
- observabilidad,
- calidad de taxonomía.

Esto es MUY importante analíticamente porque categorías sin subcategoría:
- reducen observabilidad
- dificultan análisis causal
- afectan entrenamiento NLP
- afectan routing
- afectan automation targeting.

In [129]:
left_blank_df = subcategories_df[
    subcategories_df["sub_category"]
    == "Left Blank"
]

left_blank_summary = (
    left_blank_df
    .groupby("contact_reason")["volume"]
    .sum()
    .reset_index()
    .sort_values(by="volume", ascending=False)
)

left_blank_summary

,contact_reason,volume
2,Customer Feedback,77430
10,Returns & Refunds,47749
11,Spam/No Contact,42890
13,Vouchers & Gift cards,8312
3,Defective Returns Management,7031
5,Membership,6013
1,Company Information,3776
4,Existing Order,2718
0,Apps & Website,1171
9,Product Information,317


In [130]:
fig = px.bar(
    left_blank_summary,
    x="contact_reason",
    y="volume",
    color="contact_reason",
    title="Left Blank Subcategory Analysis",
    text_auto=".2s",
)

fig.update_layout(
    xaxis_title="Contact Reason",
    yaxis_title="Volume",
    template="plotly_white",
    xaxis_tickangle=-35,
    showlegend=False,
)

fig.show()

### Interpretación — Análisis de Sub-categorías "Left Blank"

La visualización cuantifica el volumen de contactos donde la **subcategoría no fue registrada**, revelando los principales puntos ciegos de observabilidad operacional del sistema:

| Contact Reason | Vol. "Left Blank" | % del Total LB | % dentro de la categoría |
| :--- | ---: | :---: | :---: |
| Customer Feedback | **77,430** | **39.2%** | ~98.1% |
| Returns & Refunds | **47,749** | **24.2%** | ~19.5% |
| Spam / No Contact | **42,890** | **21.7%** | ~96.8% |
| Vouchers & Gift cards | **8,312** | **4.2%** | ~84.0% |
| Defective Returns Mgmt | **7,031** | **3.6%** | ~83.5% |
| Membership | **6,013** | **3.0%** | ~40.1% |
| Resto de categorías | **4,081** | **2.1%** | < 20% |
| **Total** | **197,506** | **100%** | **24.7%** del total de contactos |

$$\text{Observability Gap} = \frac{197{,}506}{798{,}617} \approx \mathbf{24.7\%}$$

**Casi 1 de cada 4 contactos no tiene subcategoría registrada.**

#### A. Customer Feedback — El Mayor Punto Ciego del Sistema

Con **77,430 contactos sin subcategoría** (~98.1% de la categoría entera), `Customer Feedback` es prácticamente opaca desde una perspectiva analítica. Dos explicaciones posibles:

1. **Taxonomía insuficiente:** No existen subcategorías definidas para capturar los motivos reales del feedback (calidad del producto, experiencia de entrega, atención recibida, etc.).
2. **Ausencia de enforcement:** Los agentes cierran el contacto sin completar el campo porque el sistema no lo exige como obligatorio.

En cualquier caso, es un problema de **data governance operacional**, no de NLP.

#### B. Spam / No Contact — Ruido que Distorsiona el Indicador

`Spam/No Contact` con 42,890 "Left Blank" (~96.8% de la categoría) representa contactos sin interacción real. Su alto volumen en esta categoría es esperable —no existe un intent que clasificar—, pero infla artificialmente el denominador del gap de observabilidad. **Es recomendable excluir esta categoría en análisis futuros de calidad de clasificación** para no distorsionar las métricas.

#### C. Returns & Refunds — El Gap con Mayor Impacto Operacional Real

Con **47,749 contactos sin subcategoría**, `Returns & Refunds` tiene el mayor impacto operacional real de todos los `Left Blank`, porque:
- Es una categoría **transaccional de alto volumen** (245,084 contactos totales).
- La subcategoría determina el flujo de resolución (return label, refund status, how to return, etc.).
- Sin subcategoría, no es posible hacer routing inteligente ni targeting de automatización.

#### D. Distinción Conceptual Crítica

| Indicador | Qué mide | Origen del Problema |
| :--- | :--- | :--- |
| `Not defined by Bot` | El bot no pudo **categorizar** el intent | Fallo NLP / Taxonomía técnica |
| `Left Blank` | La **subcategoría** no fue registrada por nadie | Fallo de proceso / Data governance |

El segundo es potencialmente más grave: indica que **ni el canal humano captura información de granularidad media** de forma consistente, lo que limita toda capacidad analítica posterior.

> **Conclusión de observabilidad:** `Left Blank` no es un problema de machine learning. Requiere tres intervenciones concretas: **(1)** expansión de la taxonomía de subcategorías para hacerla exhaustiva y mutuamente excluyente; **(2)** enforcement del campo como obligatorio en la plataforma de gestión de contactos; **(3)** validación de calidad de datos en tiempo real al cierre de cada interacción.

## Conclusión Preliminar — Section 10: Visualization & Exploratory Analytics

Las siete visualizaciones construidas en esta sección producen una **narrativa analítica cohesiva** que permite pasar de datos operacionales agregados a un diagnóstico ejecutivo accionable y cuantitativamente justificado.

### Resumen Consolidado de Hallazgos Visuales

| Sub-sección | Visualización | Hallazgo Principal |
| :--- | :--- | :--- |
| 10.2 | Channel Distribution | Bot participa en 50.1% de contactos pero solo contiene 15.5%. Tasa de contención interna del bot: **30.8%** |
| 10.3 | Top Intents Overall | Top 2 categorías = **64.6%** del volumen. Alta concentración en procesos post-compra determinísticos. |
| 10.4 | Stacked Channel View | Customer Feedback y Support/Ordering muestran dominancia absoluta de `agent_only` → fallas de diseño de flujo. |
| 10.5 | Pareto Analysis | 4 de 19 categorías = **81.7%** del volumen. Roadmap concentrado, no disperso. |
| 10.6 | Automation Opportunity | 278,926 contactos en intents de alta automatizabilidad. Potencial estimado: **+5 a +7pp** en Containment Rate. |
| 10.7 | Undefined Intents | 5,855 contactos (82.5% en `bot_only`). Falsa contención sin resolución → prerequisito técnico urgente. |
| 10.8 | Left Blank Analysis | **197,506 contactos** (24.7%) sin subcategoría. Problema de data governance que limita todo análisis posterior. |


### Árbol Causal: De los Hallazgos a los KPIs

![Árbol Causal De los Hallazgos a los KPIs.png](<attachment:data\images\Árbol Causal De los Hallazgos a los KPIs.png>)


### Priorización Basada en Evidencia

| # | Acción | Impacto Estimado | Plazo |
| :---: | :--- | :--- | :---: |
| **1** | Fix `Left Blank` → rediseño de taxonomía + enforcement de subcategoría obligatoria | Observabilidad + Resolution Rate +5pp | 0–30 días |
| **2** | Activar `Explain how to order` (56k vol., ~0% bot-rate, sin dependencias técnicas) | Containment Rate +2pp | 0–30 días |
| **3** | Fix routing `How to return` (flujo existe, hay fuga de escalación) | Containment Rate +1pp | 0–30 días |
| **4** | Integrar OMS: `Return status` + `In transit` + `label Request` (~120k vol. non-bot) | Resolution Rate +8pp, Repeat Rate −4pp | 30–60 días |
| **5** | Conversión Hybrid→Bot top 5 intents (~100k contactos) | Containment Rate +3pp | 60–90 días |

### Limitaciones Analíticas Reconocidas

> Toda conclusión de esta sección opera bajo las restricciones del dataset disponible:
>
> - **Sin granularidad temporal:** No es posible detectar tendencias, estacionalidad ni degradación de performance en el tiempo.
> - **Sin resolución por sesión:** Los KPIs de Resolution Rate y Repeat Rate son *inputs* del Business Case, no calculados desde los datos. No podemos verificar si el mismo `contact_reason` tiene mayor o menor tasa de resolución real entre canales.
> - **Sin segmentación geográfica:** El análisis cubre LAM como unidad homogénea; países individuales (Colombia, México, Brasil) pueden mostrar perfiles de escalamiento significativamente distintos.
> - **Sin datos de costo:** La cuantificación del ahorro económico requiere Average Handling Time (AHT) y costo por interacción, no disponibles en este dataset.

A pesar de estas restricciones, los datos agregados son **suficientes para construir un roadmap de optimización priorizado y cuantitativamente justificado** — que es el objetivo central de este Business Case.

# SECTION 11 — Strategic Recommendations

Basado en el análisis profundo del dataset operacional y la interpretación de los KPIs, a continuación se presentan las recomendaciones estratégicas accionables. El objetivo es estructurar un plan claro que aborde directamente las brechas identificadas en **Containment Rate (48% vs 55%)**, **Repeat Rate (28% vs 18%)** y **Resolution Rate (30% vs 50%)**.

## 11.1 Executive Storyline (Resumen Ejecutivo)

**1. Diagnóstico del Estado Actual:**
El bot cuenta con un alto nivel de participación en la interacción inicial con los usuarios (~50% de los contactos totales). Sin embargo, la **Contención Interna Real del Bot es solo del 30.8%**. Esto demuestra que el problema principal no es la adopción del canal, sino la **fuga de casos híbridos (Escalation Leakage)**, en donde el bot identifica la necesidad pero carece de la capacidad transaccional para completarla *end-to-end*, forzando la transferencia a un agente.

**2. Priorización Basada en Pareto:**
Aproximadamente el **81.7%** de los contactos se concentra en solo 4 macro-categorías, dominadas por `Existing Order` y `Returns & Refunds`. Estas interacciones son deterministas, basadas en flujos estructurados, y no requieren empatía ni juicio humano complejo, lo que las hace los candidatos ideales para la automatización transaccional.

**3. Impacto en el Business Case:**
Al automatizar eficientemente las intenciones post-compra identificadas (ej. `Return status`, `In transit`, `label Request`), existe el potencial de resolver y contener ~67,000 contactos que actualmente se escalan. Esto proyecta un **incremento directo de +5 a +7pp en el Containment Rate**, cerrando la brecha del objetivo del 55%.

---

## 11.2 Actionable Roadmap: Plan de Implementación (30 / 60 / 90 Días)

Este roadmap no está estructurado al azar; se fundamenta directamente en la Matriz de Complejidad y en el Principio de Pareto. La regla de oro en el desarrollo de producto es: *"Lanzar primero lo que genera mayor impacto con el menor esfuerzo operativo"*.

### 🔴 Fase 1: Quick Wins & Data Governance (0-30 Días)
**Enfoque:** Problemas de Complejidad Baja y gobernanza de datos.
**Impacto en KPIs:** Incremento rápido de +2 a +3pp en Containment Rate y mejora radical de observabilidad.

* **Data Governance (`Left Blank`):** Solucionar de inmediato la falta de taxonomía haciendo obligatoria la captura de la subcategoría por parte de los agentes. Esto reduce el 24.7% de punto ciego operacional.
* **FAQ Automation (`Explain how to order`):** Activar o corregir el flujo conversacional informativo de esta subcategoría (56k contactos), que no requiere integración técnica con sistemas core.
* **Routing Fix (`How to return`):** El bot ya sabe gestionar esta intención en `bot_only`, pero presenta fugas masivas hacia canales híbridos y humanos. Se deben reajustar las reglas de derivación.

> 🎙️ **Argumento Ejecutivo para Presentación:**
> *"En el primer mes no vamos a escribir una sola línea de código complejo; vamos a arreglar las reglas de enrutamiento y a activar las preguntas frecuentes. Esto nos dará un salto inmediato en el Containment Rate a costo técnico cero."*

### 🟡 Fase 2: Core Transactional Integration (30-60 Días)
**Enfoque:** Complejidad Media del Pareto (`Existing Order` y `Returns & Refunds`).
**Impacto en KPIs:** +8pp en Resolution Rate y reducción directa de la Repeat Rate.

* **OMS / Carrier Integrations:** Implementar conexiones vía API con el sistema de manejo de órdenes (OMS) y la plataforma de envíos.
* **Automatización de Alto Valor:** Construir los flujos transaccionales *end-to-end* para `Return status`, `In transit`, y `label Request`, permitiendo al bot consultar y dar respuesta en tiempo real sin fricciones.

> 🎙️ **Argumento Ejecutivo para Presentación:**
> *"Durante el segundo mes, el equipo de ingeniería se dedicará a construir las integraciones API con nuestro OMS. Esto permitirá que el bot no solo 'salude' al cliente, sino que resuelva transacciones de rastreo de pedidos y reembolsos de inicio a fin. Esto es lo que cerrará la brecha de escalamiento híbrido (Escalation Leakage) y nos hará alcanzar la meta del 55% de contención."*

### 🟢 Fase 3: AI Agent Consolidation & Long-tail (60-90 Días)
**Enfoque:** Errores del NLP (`Not defined by bot`) y la "cola larga" de 15 categorías pequeñas.
**Impacto en KPIs:** Cierre total de brechas y modernización de la experiencia.

* **Expansión Taxonómica:** Re-entrenar el motor NLP para capturar las intenciones no clasificadas, deteniendo la falsa contención.
* **Transición a AI Agent:** Introducir capacidades generativas (LLMs) seguras como *fall-back* para resolver dudas atípicas (Long-tail) usando bases de conocimiento, en vez de transferir inmediatamente a humanos.

> 🎙️ **Argumento Ejecutivo para Presentación:**
> *"En el tercer mes, daremos el salto evolutivo. Integraremos capacidades de IA Generativa / LLMs de forma controlada como 'fall-back'. En lugar de que el bot transfiera al agente cuando no entiende algo, usará bases de conocimiento internas para intentar resolver esas consultas inusuales, garantizando una modernización total de la experiencia."*

---

## 11.3 Architecture Recommendations (Estrategia de AI Agent)

Para escalar del 48% al >55% de contención de forma sostenible, recomendamos migrar gradualmente de un modelo rígidamente determinístico a una **Arquitectura de Agentes de IA (AI Agent Architecture)**:

1.  **Orquestación con LLM:** Utilizar LLMs para entender el contexto complejo y extraer entidades de las frases de los usuarios (ej. número de orden, producto), reduciendo la tasa de "Not defined".
2.  **Tools / APIs Integradas:** El Agente de IA debe estar equipado con herramientas de integración nativa. En lugar de generar respuestas textuales especulativas, el Agente debe ejecutar llamadas a las APIs del OMS, de Pagos y de Devoluciones.
3.  **Graceful Escalation (Escalamiento Inteligente):** Cuando el Agente falla o requiere intervención humana, no debe simplemente derivar. Debe empacar el resumen del contexto, el análisis de sentimiento y los metadatos transaccionales y enviarlos al CRM del agente. Esto disminuye drásticamente el *Average Handling Time* (AHT) del agente humano.

---

## 11.4 Power BI Structure & Dashboard Recommendations

Para monitorear el éxito del roadmap y garantizar visibilidad operativa, recomendamos desplegar un dashboard gerencial basado en la regla de **3-30-300 segundos** y las mejores prácticas visuales para Business Intelligence.

### 💡 Arquitectura del Dashboard (Wireframe Conceptual)

**Nivel 1: Executive Snapshot (3 segundos)**
Ubicado en la franja superior. Diseñado para que la gerencia vea inmediatamente la salud del sistema.
* **Tarjetas de KPI (KPI Cards):** `Containment Rate (vs 55%)`, `Resolution Rate (vs 50%)`, `Repeat Rate (vs 18%)`, `Total Volume`.
* Utilizar formato de semáforo (Rojo/Verde) exclusivamente para las varianzas respecto a la meta (Target Variance).

**Nivel 2: Trend & Leakage Analysis (30 segundos)**
Zona central. Orientado a responsables de operaciones de CX y dueños de producto.
* **Gráfico de Cascada (Waterfall Chart):** Muestra cómo fluye el volumen inicial desde "Total Contacts" → fuga hacia "Human Routed" → caída en "Hybrid Escalation" → resultado en "Bot Contained". Esto evidencia dónde se rompe el embudo conversacional.
* **Gráfico de Barras Apiladas 100%:** Para visualizar la proporción de `Agent_Only`, `Hybrid`, y `Bot_Only` de las 5 categorías más importantes (Top Intents).

**Nivel 3: Deep Dive Analytics (300 segundos)**
Sección inferior o en páginas de *drill-through* separadas. Para analistas y supervisores de QA.
* **Matriz de Intents Dinámica:** Tabla detallada estilo *drill-down* (`Contact Reason` > `Subcategory` > `Handling Channel`), incluyendo volúmenes absolutos y porcentajes.
* **Undefined & Blank Intent Monitor:** Un panel secundario para rastrear específicamente las fallas sistémicas (`Left Blank` y `Not defined by bot`) y garantizar que las fases 1 y 3 del roadmap mantengan el *compliance* de datos.

**Directrices Visuales:**
* Aplicar una paleta monocromática u orientada a la marca (colores corporativos de Adidas) con un color acento claro (ej. Naranja) para destacar alertas o desvíos.
* Evitar por completo gráficos de pastel (pie/donut charts) con más de 3 dimensiones debido a su ineficiencia perceptual; priorizar gráficos de barras horizontales ordenados por volumen.